## Step 1: Setup and Project Structure



In [1]:
!pip install -U langchain langchain-openai python-dotenv

In [2]:
import os
import random
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

load_dotenv()

# Make sure OPENAI_API_KEY is set in your .env file
print("OpenAI API Key loaded:", "OPENAI_API_KEY" in os.environ)

model = init_chat_model(model="gpt-4o-mini", model_provider="openai", temperature=0.7)

OpenAI API Key loaded: True


## Step 2: Create Creative Tools


Step 2a: The Demogorgon tool (provided)

In [3]:
@tool
def consult_demogorgon(complaint: str) -> str:
    """Consult the Demogorgon about a complaint involving the Upside Down.
    Returns a chaotic, unreliable opinion."""
    responses = [
        f"The Demogorgon sniffs '{complaint}' and roars. It seems... hungry, not helpful.",
        f"The Demogorgon tilts its petal-face at '{complaint}'. Interpreted as: 'turn off the lights'.",
        f"The Demogorgon ignores '{complaint}' and walks back into the wall. Classic.",
    ]
    return random.choice(responses)

In [18]:
print(consult_demogorgon.invoke({"complaint": "My lights keep flickering"}))

The Demogorgon ignores 'My lights keep flickering' and walks back into the wall. Classic.


Step 2b: The Hawkins records tool

In [19]:
@tool
def check_hawkins_records(query: str) -> str:
    """Search the official Hawkins records for facts.
    Use this for reliable information about: portal, monsters, psychics, electricity.
    Returns the matching record, or 'No record found' if nothing matches."""
    records = {
        "portal": "Record P-01: The portal in the Hawkins lab basement opened after a psychic experiment. It grows when electrical activity is high.",
        "monsters": "Record M-07: Demogorgons and Demodogs hunt at night and are drawn to blood and fear. Fire and bright light drive them back.",
        "psychics": "Record S-11: Subject Eleven can move objects with her mind. Using too much power causes nosebleeds and exhaustion.",
        "electricity": "Record E-03: Lights flicker and power fails when creatures from the Upside Down are nearby.",
    }

    query_lower = query.lower()            # make the search case-insensitive

    for key, record in records.items():
        if key in query_lower:             # is the keyword inside the question?
            return record

    return f"No record found in the Hawkins archive for: '{query}'"

In [20]:
print(check_hawkins_records.invoke({"query": "tell me about the portal"}))
print(check_hawkins_records.invoke({"query": "my neighbor's cat is weird"}))

Record P-01: The portal in the Hawkins lab basement opened after a psychic experiment. It grows when electrical activity is high.
No record found in the Hawkins archive for: 'my neighbor's cat is weird'


Step 2c: The spell tool

In [22]:
@tool
def cast_interdimensional_spell(problem: str, creativity_level: str = "medium") -> str:
    """Cast creative interdimensional spells to solve a problem.
    creativity_level can be 'low' (1 spell), 'medium' (2 spells) or 'high' (3 spells).
    Returns fun, made-up solutions, not facts."""
    levels = {"low": 1, "medium": 2, "high": 3}
    n = levels.get(creativity_level.lower(), 2)      # unknown level → medium

    spells = [
        f"Spell of Flickering Lights: string Christmas lights around '{problem}' and ask it yes/no questions.",
        f"Walkie-Talkie Ward: radio the party on channel 4 and describe '{problem}' in code words.",
        f"Waffle Offering: leave a plate of waffles near '{problem}' and wait one night.",
        f"Mixtape Shield: play your favorite song loudly at '{problem}' until it backs away.",
    ]

    chosen = random.sample(spells, min(n, len(spells)))   # pick n different spells
    return "\n".join(chosen)

In [24]:
print(cast_interdimensional_spell.invoke({"problem": "Demodogs in my garden", "creativity_level": "low"}))

Walkie-Talkie Ward: radio the party on channel 4 and describe 'Demodogs in my garden' in code words.


Step 2d: The party wisdom tool

In [26]:
@tool
def gather_party_wisdom(question: str) -> str:
    """Ask the party (Mike, Dustin, Lucas, Will) for advice.
    Topics they know: plan, science, danger, upside down.
    Returns the advice of the matching party member, or a general group answer."""
    wisdom = {
        "plan": "Mike: 'We stick together, we make a plan at my basement, and nobody goes in alone.'",
        "science": "Dustin: 'Science says: find out what it eats and what it's afraid of. Then use it.'",
        "danger": "Lucas: 'Bring the slingshot, keep a way out, and don't trust anything that looks friendly.'",
        "upside down": "Will: 'If it feels cold and you sense it watching you, it's from the Upside Down. Get out.'",
    }

    question_lower = question.lower()

    for key, advice in wisdom.items():
        if key in question_lower:
            return advice

    return "The party huddles in Mike's basement... and agrees: stay together, stay calm, and call Eleven."

In [27]:
print(gather_party_wisdom.invoke({"question": "what is the plan?"}))
print(gather_party_wisdom.invoke({"question": "how do I bake a cake?"}))

Mike: 'We stick together, we make a plan at my basement, and nobody goes in alone.'
The party huddles in Mike's basement... and agrees: stay together, stay calm, and call Eleven.


## Step 3: Create Agent with Tools


In [28]:
tools = [
    consult_demogorgon,
    check_hawkins_records,
    cast_interdimensional_spell,
    gather_party_wisdom,
]

system_prompt = """You are the creative complaint handler for NormalObjects.

How to work:
1. Always call at least one tool before you answer. Never answer from your own knowledge.
2. Be creative: combine tools in any order that helps the customer.
3. For facts, use check_hawkins_records. Treat the Demogorgon and spells as fun, not as facts.
4. Answer ONLY from the tool results. If a tool says "No record found", say so honestly. Do not invent records.

At the end of your answer, add a line:
Tools used: <list of the tools you called>"""

agent = create_agent(model=model, tools=tools, system_prompt=system_prompt)

print([t.name for t in tools])

['consult_demogorgon', 'check_hawkins_records', 'cast_interdimensional_spell', 'gather_party_wisdom']


## Step 4: Test with Sample Complaints


In [29]:
complaints = [
    "My Christmas lights keep flickering and I hear noises from the wall. What is going on?",
    "There is a weird portal in my basement. What's the plan?",
    "A Demodog ate my homework. I need a creative solution, creativity level high!",
    "My neighbor's cat is acting weird. Is it from the Upside Down?",  # no record exists
]

results = []

for complaint in complaints:
    print("=" * 70)
    print("COMPLAINT:", complaint)

    result = agent.invoke(
        {"messages": [HumanMessage(content=complaint)]},
        config={"recursion_limit": 10},
    )
    results.append(result)

    print("\nFINAL ANSWER:")
    print(result["messages"][-1].content)

print("=" * 70)
print(f"Done: {len(results)} complaints handled")

COMPLAINT: My Christmas lights keep flickering and I hear noises from the wall. What is going on?

FINAL ANSWER:
It seems that there are no records in the Hawkins archive regarding flickering lights, and the Demogorgon has given a rather ominous response, suggesting it might be hungry—definitely not the help you were looking for!

However, I have some creative spells to help you tackle this issue:
1. **Walkie-Talkie Ward**: Radio the party on channel 4 and describe the situation of the flickering lights and noises using code words.
2. **Spell of Flickering Lights**: Wrap the Christmas lights around the source of the flickering and ask it yes/no questions to get to the root of the problem.
3. **Waffle Offering**: Leave a plate of waffles near the source of the flickering lights and noises, and wait overnight to see if it resolves the issue!

Give these a try and see if they help bring some holiday cheer back into your home!

Tools used: check_hawkins_records, consult_demogorgon, cast_in

## Step 5: Analyze Agent Behavior

Step 5a: Track tool usage

In [30]:
from collections import Counter

all_tools_used = []

for complaint, r in zip(complaints, results):
    order = []
    for msg in r["messages"]:
        # AI messages can contain tool requests
        for call in getattr(msg, "tool_calls", []) or []:
            order.append(call["name"])

    all_tools_used.extend(order)
    print(complaint[:50] + "...")
    print("   ", " → ".join(order) if order else "(no tools)")
    print()

print("TOOL COUNTS:")
for name, count in Counter(all_tools_used).most_common():
    print(f"   {name}: {count}")

My Christmas lights keep flickering and I hear noi...
    check_hawkins_records → consult_demogorgon → cast_interdimensional_spell

There is a weird portal in my basement. What's the...
    check_hawkins_records → gather_party_wisdom

A Demodog ate my homework. I need a creative solut...
    cast_interdimensional_spell

My neighbor's cat is acting weird. Is it from the ...
    check_hawkins_records → consult_demogorgon → cast_interdimensional_spell

TOOL COUNTS:
   check_hawkins_records: 3
   cast_interdimensional_spell: 3
   consult_demogorgon: 2
   gather_party_wisdom: 1


Step 5b: Trace one answer (grounding)

In [31]:
r = results[1]   # the portal complaint

for msg in r["messages"]:
    if msg.type == "human":
        print("👤 COMPLAINT:", msg.content)

    elif msg.type == "ai" and msg.tool_calls:
        for call in msg.tool_calls:
            print(f"\n🤖 AGENT CALLS: {call['name']}")
            print(f"   args: {call['args']}")

    elif msg.type == "tool":
        print(f"\n🔧 TOOL RETURNED ({msg.name}):")
        print("  ", msg.content)

print("\n✅ FINAL ANSWER:")
print(r["messages"][-1].content)

👤 COMPLAINT: There is a weird portal in my basement. What's the plan?

🤖 AGENT CALLS: check_hawkins_records
   args: {'query': 'portal'}

🤖 AGENT CALLS: gather_party_wisdom
   args: {'question': 'What should I do about a weird portal in my basement?'}

🔧 TOOL RETURNED (check_hawkins_records):
   Record P-01: The portal in the Hawkins lab basement opened after a psychic experiment. It grows when electrical activity is high.

🔧 TOOL RETURNED (gather_party_wisdom):
   The party huddles in Mike's basement... and agrees: stay together, stay calm, and call Eleven.

✅ FINAL ANSWER:
According to the official Hawkins records, portals can be quite unpredictable. Notably, Record P-01 indicates that a portal opened in the Hawkins lab basement after a psychic experiment, and they can grow when electrical activity is high. 

As for the advice from the party, they suggest that you should stay together and stay calm, and definitely call Eleven for help. 

Make sure to gather your friends and keep a cl

Step 5c: Compare with a structured handler

In [32]:
def structured_handler(complaint: str) -> str:
    """Fixed order, no LLM: records → party → spell (low)."""
    record = check_hawkins_records.invoke({"query": complaint})
    advice = gather_party_wisdom.invoke({"question": complaint})
    spell = cast_interdimensional_spell.invoke({"problem": complaint, "creativity_level": "low"})

    return (
        f"COMPLAINT: {complaint}\n\n"
        f"1. HAWKINS RECORDS:\n   {record}\n\n"
        f"2. PARTY WISDOM:\n   {advice}\n\n"
        f"3. SPELL (low):\n   {spell}"
    )

print(structured_handler(complaints[1]))

COMPLAINT: There is a weird portal in my basement. What's the plan?

1. HAWKINS RECORDS:
   Record P-01: The portal in the Hawkins lab basement opened after a psychic experiment. It grows when electrical activity is high.

2. PARTY WISDOM:
   Mike: 'We stick together, we make a plan at my basement, and nobody goes in alone.'

3. SPELL (low):
   Spell of Flickering Lights: string Christmas lights around 'There is a weird portal in my basement. What's the plan?' and ask it yes/no questions.


| | Agent | Structured |
|---|---|---|
| Same tools every run? | No: each complaint got a different set (1 to 3 tools) | Yes: always records → party → spell |
| Tool order predictable? | No: the agent decides the order | Yes: fixed order |
| Can invent facts? | Yes: e.g. "portals can be quite unpredictable", and it treated the Demogorgon's random answer as "the key" | No: the output is only tool text, nothing added |
| Easy to trace? | Only by digging through messages (tool_calls + tool outputs) | Yes: every line comes from one named tool |
| Best for which client case? | Creative, friendly answers where small inventions are okay (e.g. fun marketing chatbot) | Cases that need consistency and an audit trail (e.g. support, compliance) |